# Store Sales – Temporal Fusion Transformer (TFT)

The neural challenger: where our LightGBM sees only hand-built feature columns, the TFT consumes the **raw recent history as a sequence** (learning its own lag-like representations via attention) plus the three covariate types it was designed for:

- **static** — store & family identity (per-series metadata);
- **known future** — promotions, holidays, paydays, calendar, oil (values exist for the forecast days);
- **observed past** — the sales sequence itself.

Same target scale (`log1p` → MSE loss = RMSLE), same holdout (2017-07-31 → 2017-08-15), so the score is directly comparable with `03_lightgbm.ipynb` (pure LightGBM ≈ 0.398) and `04_hybrid_ensemble.ipynb`.

**Expectations, set honestly:** on tabular retail data with rich covariates, tuned GBDTs usually win (cf. the M5 competition). A modestly-sized TFT trained for a bounded number of steps on a laptop is *not* expected to beat 0.398 — the point is to see how close the learn-your-own-features paradigm gets, at what cost.

Training here is configured to stay laptop-sized: data from 2015+, hidden size 64, ≤1,000 optimizer steps with early stopping.

## 0. Setup

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # conda libomp + pip-torch libomp coexistence (macOS)

import numpy as np
import pandas as pd
import torch
from pathlib import Path

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MSE

PROC = Path('../data/processed')
train = pd.read_pickle(PROC / 'train_features.pkl')
test = pd.read_pickle(PROC / 'test_features.pkl')

VAL_START = pd.Timestamp('2017-07-31')
HIST_START = pd.Timestamp('2015-01-01')   # keep training laptop-sized

def rmsle(y_true, y_pred):
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

print('torch', torch.__version__, '| mps:', torch.backends.mps.is_available())

/Users/olaf/conda/envs/learn-data-science/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-12 21:40:56,766	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-06-12 21:40:56,925	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


torch 2.12.0 | mps: True


## 1. Reshape to NeuralForecast's long format

One row per (series, day): `unique_id` (store_family), `ds` (date), `y` (`log1p` sales) + exogenous columns. Two details:

- the 2 missing Christmas days (2015/2016) are reinserted with `y = 0` — NeuralForecast needs gap-free daily series;
- static metadata (store, family, cluster, type) goes in a separate `static_df`, numerically encoded.

In [2]:
FUTR = ['onpromotion', 'promo_roll7_mean', 'oil_price', 'dayofweek',
        'day_of_month', 'is_payday', 'is_holiday', 'is_event']

def to_nf(df, with_y=True):
    out = pd.DataFrame({
        'unique_id': df['store_nbr'].astype(str) + '_' + df['family'].astype(str),
        'ds': df['date'],
    })
    if with_y:
        out['y'] = df['sales_log1p'].astype('float32')
    for c in FUTR:
        out[c] = df[c].astype('float32')
    return out

df = to_nf(train[train['date'] >= HIST_START])

# fill the two missing Christmas days per series (stores closed -> y = 0)
full_idx = pd.MultiIndex.from_product(
    [df['unique_id'].unique(),
     pd.date_range(HIST_START, train['date'].max(), freq='D')],
    names=['unique_id', 'ds'])
df = df.set_index(['unique_id', 'ds']).reindex(full_idx).reset_index()
# young stores legitimately start late: forward gaps only exist before their first
# sale -- drop those, fill the true in-series gaps (Christmas) with closed-day values
first_obs = df.dropna(subset=['y']).groupby('unique_id')['ds'].min().rename('first_obs')
df = df.merge(first_obs, on='unique_id')
df = df[df['ds'] >= df['first_obs']].drop(columns='first_obs')
df[['y'] + FUTR] = df[['y'] + FUTR].fillna(0)

static_df = (train[['store_nbr', 'family', 'cluster', 'store_type']]
             .assign(unique_id=lambda d: d['store_nbr'].astype(str) + '_' + d['family'].astype(str))
             .drop_duplicates('unique_id'))
static_df = pd.DataFrame({
    'unique_id': static_df['unique_id'],
    'store_code': static_df['store_nbr'].cat.codes.astype('float32'),
    'family_code': static_df['family'].cat.codes.astype('float32'),
    'cluster_code': static_df['cluster'].cat.codes.astype('float32'),
    'type_code': static_df['store_type'].cat.codes.astype('float32'),
})

print(f'{df["unique_id"].nunique()} series, {len(df):,} rows, '
      f'{df["ds"].min().date()} -> {df["ds"].max().date()}')

1782 series, 1,651,782 rows, 2015-01-01 -> 2017-08-15


## 2. Train on data before the holdout

`input_size=56`: each training window shows the model 8 weeks of raw history to predict the next 16 days. `scaler_type='robust'` normalizes each window by its own scale — that's how the net handles series of wildly different sizes (and trends) without absolute clocks.

In [3]:
trn_df = df[df['ds'] < VAL_START]

def make_tft():
    return TFT(
        h=16, input_size=56,
        hidden_size=64, n_head=4,
        loss=MSE(),                      # MSE on log1p target = RMSLE
        learning_rate=1e-3,
        max_steps=1000,
        batch_size=64, windows_batch_size=512,
        val_check_steps=50, early_stop_patience_steps=5,
        scaler_type='robust',
        futr_exog_list=FUTR,
        stat_exog_list=['store_code', 'family_code', 'cluster_code', 'type_code'],
        random_seed=42,
    )

nf = NeuralForecast(models=[make_tft()], freq='D')
nf.fit(df=trn_df, static_df=static_df, val_size=16)

Seed set to 42


GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores



  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MSE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 1.7 K  | train
4 | static_encoder          | StaticCovariateEncoder   | 160 K  | train
5 | temporal_encoder        | TemporalCovariateEncoder | 457 K  | train
6 | temporal_fusion_decoder | TemporalFusionDecoder    | 64.8 K | train
7 | output_adapter          | Linear                   | 65     | train
-----------------------------------------------------------------------------
684 K     Trainable params
0         Non-trainable params
684 K     Total params
2.739     Total estimated model params size (MB)
278       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation DataLoader 0: 100%|██████████| 28/28 [00:00<00:00, 37.25it/s]

Epoch 35: 100%|██████████| 20/20 [00:03<00:00,  5.85it/s, v_num=0, train_loss_step=2.600, train_loss_epoch=3.400, valid_loss=0.167]

Epoch 35: 100%|██████████| 20/20 [00:03<00:00,  5.85it/s, v_num=0, train_loss_step=2.600, train_loss_epoch=3.160, valid_loss=0.167]

`Trainer.fit` stopped: `max_steps=1000` reached.


Epoch 35: 100%|██████████| 20/20 [00:03<00:00,  5.84it/s, v_num=0, train_loss_step=2.600, train_loss_epoch=3.160, valid_loss=0.167]

## 3. Predict the holdout & score

In [4]:
val = train[train['date'] >= VAL_START]
futr_val = to_nf(val, with_y=False)

pred = nf.predict(futr_df=futr_val)
pred_col = [c for c in pred.columns if c not in ('unique_id', 'ds')][0]

eva = (val.assign(unique_id=val['store_nbr'].astype(str) + '_' + val['family'].astype(str))
       .rename(columns={'date': 'ds'})
       [['unique_id', 'ds', 'sales', 'is_dead_series', 'family']]
       .merge(pred, on=['unique_id', 'ds'], how='left'))
assert eva[pred_col].notna().all()

eva['pred_sales'] = np.where(eva['is_dead_series'] == 1, 0.0,
                             np.expm1(eva[pred_col]).clip(lower=0))
score_tft = rmsle(eva['sales'].to_numpy(), eva['pred_sales'].to_numpy())

print(f'TFT holdout RMSLE: {score_tft:.5f}')
print('reference: pure LightGBM 0.39847  |  naive rolling-28 baseline 0.55023')

eva['sq_log_err'] = (np.log1p(eva['pred_sales']) - np.log1p(eva['sales'])) ** 2
worst = (eva.groupby('family', observed=True)['sq_log_err'].mean() ** 0.5).sort_values()
print('\n5 worst families:')
print(worst.tail(5).round(4).to_string())

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.


GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


/Users/olaf/conda/envs/learn-data-science/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

...

Predicting DataLoader 0:  93%|█████████▎| 26/28 [00:00<00:00, 45.55it/s]

Predicting DataLoader 0:  96%|█████████▋| 27/28 [00:00<00:00, 45.60it/s]

Predicting DataLoader 0: 100%|██████████| 28/28 [00:00<00:00, 45.54it/s]

Predicting DataLoader 0: 100%|██████████| 28/28 [00:00<00:00, 45.29it/s]

TFT holdout RMSLE: 0.48096
reference: pure LightGBM 0.39847  |  naive rolling-28 baseline 0.55023

5 worst families:
family
BEAUTY                        0.5570
CELEBRATION                   0.5589
GROCERY II                    0.6105
LINGERIE                      0.6700
SCHOOL AND OFFICE SUPPLIES    1.5648


## 4. Submission (optional comparison point)

Refit on **all** data through 2017-08-15 with the same budget, forecast the real test window, write `submission_tft.csv`. Worth uploading once to see where a laptop-budget TFT lands on the leaderboard relative to 0.44810.

In [5]:
nf_full = NeuralForecast(models=[make_tft()], freq='D')
nf_full.fit(df=df, static_df=static_df, val_size=16)

futr_test = to_nf(test, with_y=False)
pred_test = nf_full.predict(futr_df=futr_test)
pcol = [c for c in pred_test.columns if c not in ('unique_id', 'ds')][0]

sub = (test.assign(unique_id=test['store_nbr'].astype(str) + '_' + test['family'].astype(str))
       .rename(columns={'date': 'ds'})
       [['id', 'unique_id', 'ds', 'is_dead_series']]
       .merge(pred_test, on=['unique_id', 'ds'], how='left'))
assert sub[pcol].notna().all()
sub['sales'] = np.where(sub['is_dead_series'] == 1, 0.0, np.expm1(sub[pcol]).clip(lower=0))

sub[['id', 'sales']].sort_values('id').to_csv('submission_tft.csv', index=False)
print(f'submission_tft.csv written, sales range {sub["sales"].min():.2f} -> {sub["sales"].max():.2f}')

Seed set to 42


GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores



  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MSE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 1.7 K  | train
4 | static_encoder          | StaticCovariateEncoder   | 160 K  | train
5 | temporal_encoder        | TemporalCovariateEncoder | 457 K  | train
6 | temporal_fusion_decoder | TemporalFusionDecoder    | 64.8 K | train
7 | output_adapter          | Linear                   | 65     | train
-----------------------------------------------------------------------------
684 K     Trainable params
0         Non-trainable params
684 K     Total params
2.739     Total estimated model params size (MB)
278       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Predicting DataLoader 0:  93%|█████████▎| 26/28 [00:00<00:00, 45.59it/s]

Predicting DataLoader 0:  96%|█████████▋| 27/28 [00:00<00:00, 45.74it/s]

Predicting DataLoader 0: 100%|██████████| 28/28 [00:00<00:00, 45.78it/s]

Predicting DataLoader 0: 100%|██████████| 28/28 [00:00<00:00, 45.63it/s]

submission_tft.csv written, sales range 0.00 -> 15265.34


## Reading the result

Three questions this notebook answers:

1. **Does the no-feature-engineering paradigm work at all here?** Compare with the naive baseline (0.550) — the TFT should beat it comfortably purely from raw sequences + covariates.
2. **Does it match the hand-engineered GBDT (0.398)?** If it falls short, that gap is the measured value of our 35 columns of domain knowledge (within this compute budget — a bigger TFT, tuned longer, would close some of it).
3. **Cost comparison:** LightGBM fits in ~3 minutes on CPU; this trains for substantially longer for (likely) a worse score — on *this* kind of data. The neural trade-off pays off elsewhere: huge series counts, no covariates, or when one pretrained model must serve many datasets.